In [1]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/mahasiswa/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/mahasiswa/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, row_number
from pyspark.sql.window import Window
import pandas as pd
import subprocess

# Mengambil konfigurasi default HDFS sistem secara otomatis untuk menghindari error URI
try:
    hdfs_uri = subprocess.check_output("hdfs getconf -confKey fs.defaultFS", shell=True).decode().strip()
    file_path = f"{hdfs_uri}/user/mahasiswa/tugas5/transaksi_tugas5.csv"
except:
    # Fallback jika subprocess gagal
    file_path = "hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv"

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas5_Praktikum") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 1. Membaca data dari HDFS
print(f"Mencoba membaca dari HDFS path: {file_path}")
df_transaksi = spark.read.csv(file_path, header=True, inferSchema=True)

# Menambahkan kolom pendapatan (unit_terjual dikali harga_satuan)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Membuat DataFrame df_target
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("Data berhasil dimuat. Berikut 5 baris pertama df_transaksi:")
df_transaksi.show(5)



Mencoba membaca dari HDFS path: hdfs://localhost:9000/user/mahasiswa/tugas5/transaksi_tugas5.csv
Data berhasil dimuat. Berikut 5 baris pertama df_transaksi:
+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
+--------+--------------------+----------+------------+------------+----------+
only showing top 5 rows



In [6]:
# Meringkas total pendapatan per kota
ringkasan_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan tabel target cabang
hasil_a = ringkasan_kota.join(df_target, on="kota", how="inner")

# Menambahkan kolom pencapaian_persen
hasil_a = hasil_a.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

# Mengurutkan dari pencapaian tertinggi
hasil_a = hasil_a.orderBy(col("pencapaian_persen").desc())

print("--- Hasil Bagian A ---")
hasil_a.show()

--- Hasil Bagian A ---
+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [7]:
# Agregasi total pendapatan untuk setiap kategori di masing-masing kota
pendapatan_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan_kategori")
)

# Mendefinisikan window: partisi per kota, urutkan berdasarkan pendapatan menurun
window_kategori = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

# Memberikan nomor urut menggunakan row_number()
df_ranked = pendapatan_kategori.withColumn("urutan", row_number().over(window_kategori))

# Filter hanya urutan 1 (kategori terlaris per kota)
hasil_b = df_ranked.filter(col("urutan") == 1).drop("urutan")

print("--- Hasil Bagian B ---")
hasil_b.show()

--- Hasil Bagian B ---
+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|  Magelang|Kesehatan & Kecan...|                  7275000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|  Semarang|        Rumah Tangga|                 11125000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|Yogyakarta|             Fashion|                 13325000|
+----------+--------------------+-------------------------+



In [8]:
# Mendaftarkan temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Menulis kueri SQL murni
kueri_c = """
    SELECT t.kota, tar.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tar ON t.kota = tar.kota
    GROUP BY t.kota, tar.pic_cabang
    ORDER BY jumlah_transaksi DESC
"""

hasil_c = spark.sql(kueri_c)

print("--- Hasil Bagian C ---")
hasil_c.show()

--- Hasil Bagian C ---
+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+

